# 05. Evaluation

Extraction quality cannot be assumed — it must be measured. This notebook builds an
evaluation harness from first principles, teaching the three questions every eval
system must answer: what counts as correct, how to measure it, and how to run the
measurement at scale. We then apply the harness to the archive extraction agent from
notebook 01 and interpret the results as a DH practitioner would.

## Concepts
- The eval design problem: what counts as a correct extraction?
- The three-part structure of every eval system: dataset, grader, runner
- Three match strategies: exact, substring, token F1
- Precision, recall, and F1 for set-valued fields
- Reading results to find systematic errors and design mismatches

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip`
from the repository root to the current working directory using the Files panel,
then rerun the setup cell below.

## Further reading
- Agents SDK results: https://openai.github.io/openai-agents-python/results/
- CLEF HIPE (annotated historical NER corpora): https://hipe-eval.github.io/HIPE-2022/
- SQuAD token-F1 metric: https://rajpurkar.github.io/SQuAD-explorer/

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import pandas as pd
from dotenv import load_dotenv
from agents import Agent, Runner, set_tracing_export_api_key, trace

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/.')
    return documents


DOCUMENTS = load_documents()
for doc in DOCUMENTS:
    print(doc['id'], doc['filename'])

## The hardest question in eval: what counts as correct?

Before writing any metric code, we need to settle a prior question that most eval tutorials skip:
**what does a correct extraction look like?**

Consider a letter that contains the sentence: *"I write to you from Madrid."*
The gold standard says `"Madrid"`. The agent could plausibly return any of:

| Extracted value | Verdict | Comment |
|---|---|---|
| `"Madrid"` | Correct | Exact match |
| `"Madrid, Spain"` | Debatable | More specific; not stated in the text |
| `"Comunidad de Madrid"` | Probably wrong | Administrative unit inappropriate for 1871 |
| `"the capital"` | Wrong | Correct inference but not extracted text |

None of these is obviously right or wrong in isolation.
**The match function you choose encodes your scholarly judgment.**
A strict exact-match criterion is transparent and easy to audit, but it penalises the
agent for being usefully more specific. A looser substring match is more forgiving but
risks counting genuinely wrong extractions as correct.

This design choice belongs in a methods section, not in a comment buried in a script.
We will implement three match strategies so you can see exactly what each choice costs.

In [ ]:
# Concrete illustration — no agent yet, just plain Python.
# The goal is to make the scoring stakes visible before we write any metrics.

gold_place = 'Madrid'
agent_outputs = [
    ('Madrid',               'exact match'),
    ('Madrid, Spain',        'more specific'),
    ('Comunidad de Madrid',  'administrative unit'),
    ('the capital',          'inference, not extraction'),
]

print(f'Gold: {gold_place!r}')
print()
for value, comment in agent_outputs:
    exact     = value.lower() == gold_place.lower()
    substring = gold_place.lower() in value.lower() or value.lower() in gold_place.lower()
    print(f'  {value!r:<28}  exact={exact!s:<5}  substring={substring!s:<5}  # {comment}')

## The three-part structure of every eval system

Every evaluation harness — whether you write it yourself or use a framework like
[openai/evals](https://github.com/openai/evals) — has exactly three components:

| Part | What it is | In this notebook |
|---|---|---|
| **Dataset** | Inputs + expected outputs | `GOLD` dict — manually annotated answers |
| **Grader** | Scores one prediction against one gold answer | `score_field()` with a pluggable match function |
| **Runner** | Applies the grader to every example, aggregates | the `for doc in DOCUMENTS` loop |

The `openai/evals` framework formalises this as: a JSONL dataset file, a Python or
model-graded eval spec, and a CLI runner. The structure is identical to what we build here;
only the format differs. Knowing the structure makes adopting any eval framework straightforward.

The next three sections build each part in order.

## Part 1: Dataset — the gold standard

A gold standard is a set of manually verified correct answers. In this notebook we write
them directly as a Python dict. In a real project they would come from a spreadsheet,
a labelling tool, or a peer-reviewed annotation scheme.

Two rules for writing useful gold standards in DH:

1. **Be specific about what counts as correct, in writing, before you annotate.**
   Does `"4 April 1871"` match `"April 1871"`? Decide first; never decide per-annotation.
2. **Annotate only what the text explicitly states.**
   If the letter implies a person but does not name them, leave them out.

The gold below covers five documents. `letter_06_ocr_table.txt` is intentionally omitted:
a subscriber table is not a letter, and scoring it with letter criteria would contaminate
the aggregate numbers. That is itself a design decision worth recording.

In [ ]:
GOLD: dict[str, dict[str, list[str]]] = {
    'letter_01_madrid_1871.txt': {
        'people': ['Maria Gomez'],
        'places': ['Madrid', 'Valencia'],
        'dates':  ['4 April 1871'],
    },
    'letter_02_seville_1894.txt': {
        'people': ['Antonio Ruiz'],
        'places': ['Seville'],
        'dates':  ['18 June 1894'],
    },
    # OCR fragment: annotations cover only what is legible.
    # Poor scores here reflect document quality, not agent failure.
    'letter_03_ocr_fragment.txt': {
        'people': ['Maria Gomez'],
        'places': ['Madrid'],
        'dates':  ['1871'],
    },
    'letter_04_barcelona_1902.txt': {
        'people': ['A. Castelló', 'Miquel Font', 'Ramon Vidal', 'Eduard Domènech'],
        'places': ['Barcelona', 'Girona'],
        'dates':  ['12 March 1902'],
    },
    'letter_05_porto_1888.txt': {
        'people': ['Mr. Harrington', 'Eça de Queirós', 'Isabel Ferreira', 'Rodrigues'],
        'places': ['Porto', 'Lisbon', 'Braga'],
        'dates':  ['3 September 1888'],
    },
    # Design note: gold 'dates' contains only the explicit letter date.
    # The letters also mention implicit temporal references ('last autumn', 'winter of 1878').
    # Whether those belong is a scholarly decision — not a technical one.
    # We keep the gold narrow to make the evaluation strict and interpretable,
    # and we will revisit this decision when we inspect the results.
}

print(f'Gold standard covers {len(GOLD)} documents.')
GOLD

## Part 2: Grader — match functions

The grader compares a predicted value to a gold value and returns True or False.
Three strategies cover most extraction evaluation scenarios:

| Strategy | Rule | Good for |
|---|---|---|
| **Exact** | `pred.lower() == gold.lower()` | Controlled extraction; easy to audit |
| **Substring** | `pred ⊆ gold` or `gold ⊆ pred` | Entity variants: `"Gomez"` ↔ `"Maria Gomez"` |
| **Token F1** | token overlap ≥ threshold | Date and name variants with different word order |

**Why token F1 catches date variants that substring misses:**

| Predicted | Gold | Exact | Substring | Token F1 |
|---|---|:---:|:---:|:---:|
| `"4 April 1871"` | `"April 4, 1871"` | ✗ | ✗ | ✓ (`{4, april, 1871}` shared) |
| `"18 June 1894"` | `"18 June 1894"` | ✓ | ✓ | ✓ |
| `"Gomez"` | `"Maria Gomez"` | ✗ | ✓ | ✓ |
| `"brotner"` | `"brother"` | ✗ | ✗ | ✗ (OCR error; nothing matches) |

Token F1 with a 0.5 threshold catches reformatted values without requiring explicit
normalisation rules.

In [ ]:
def is_match_exact(pred: str, gold: str) -> bool:
    return pred.strip().lower() == gold.strip().lower()


def is_match_substring(pred: str, gold: str) -> bool:
    p, g = pred.strip().lower(), gold.strip().lower()
    return p in g or g in p


def is_match_token_f1(pred: str, gold: str, threshold: float = 0.5) -> bool:
    pred_tokens = set(pred.strip().lower().split())
    gold_tokens = set(gold.strip().lower().split())
    if not pred_tokens or not gold_tokens:
        return False
    common = pred_tokens & gold_tokens
    if not common:
        return False
    p = len(common) / len(pred_tokens)
    r = len(common) / len(gold_tokens)
    f1 = 2 * p * r / (p + r)
    return f1 >= threshold

In [ ]:
# Verify the match functions against the cases from the table above.
test_cases = [
    ('4 April 1871',   '4 April 1871',   'exact date'),
    ('4 April 1871',   'April 4, 1871',  'date, different order'),
    ('Maria Gomez',    'Maria Gomez',    'exact name'),
    ('Gomez',          'Maria Gomez',    'partial name'),
    ('Madrid, Spain',  'Madrid',         'place, more specific'),
    ('brotner',        'brother',        'OCR error'),
]

hdr = f'{"Predicted":<22} {"Gold":<18} {"Exact":>6} {"Sub":>5} {"TokF1":>6}'
print(hdr)
print('-' * len(hdr))
for pred, gold, label in test_cases:
    e = is_match_exact(pred, gold)
    s = is_match_substring(pred, gold)
    t = is_match_token_f1(pred, gold)
    print(f'{pred!r:<22} {gold!r:<18} {str(e):>6} {str(s):>5} {str(t):>6}  # {label}')

## Scoring a set of extractions

Agent extraction returns a *list* of values per field, not a single value.
Scoring a list against a gold list requires counting:

- **TP** (true positive): a predicted item that matches at least one gold item
- **FP** (false positive): a predicted item that matches no gold item
- **FN** (false negative): a gold item that matches no predicted item

From these: **precision** = TP / (TP + FP), **recall** = TP / (TP + FN),
**F1** = harmonic mean of precision and recall.

The key design here: the match test is a parameter, not hard-coded equality.
Swapping `is_match_exact` for `is_match_substring` changes what counts as a TP
without touching the scoring logic.

In [ ]:
def score_field(
    predicted: list[str],
    gold: list[str],
    match_fn: Callable[[str, str], bool] = is_match_exact,
) -> dict:
    tp = sum(1 for p in predicted if any(match_fn(p, g) for g in gold))
    fp = sum(1 for p in predicted if not any(match_fn(p, g) for g in gold))
    fn = sum(1 for g in gold if not any(match_fn(p, g) for p in predicted))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        'precision': round(precision, 2),
        'recall':    round(recall, 2),
        'f1':        round(f1, 2),
        'tp': tp, 'fp': fp, 'fn': fn,
    }


@dataclass
class Extraction:
    people:   list[str] = field(default_factory=list)
    places:   list[str] = field(default_factory=list)
    dates:    list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)


def evaluate(
    extraction: Extraction,
    gold: dict,
    match_fn: Callable[[str, str], bool] = is_match_exact,
) -> pd.DataFrame:
    rows = []
    for field_name in ('people', 'places', 'dates'):
        metrics = score_field(
            getattr(extraction, field_name, []),
            gold.get(field_name, []),
            match_fn=match_fn,
        )
        rows.append({'field': field_name, **metrics})
    return pd.DataFrame(rows)


# Smoke test with a perfect prediction.
perfect = Extraction(people=['Maria Gomez'], places=['Madrid', 'Valencia'], dates=['4 April 1871'])
evaluate(perfect, GOLD['letter_01_madrid_1871.txt'])

## Part 3: Runner — apply at scale

The runner applies the grader to every document and aggregates the results.
Here it is a plain `for` loop. In `openai/evals` it is the CLI command `oaieval`;
the logic is identical.

The next two cells define the agent and run it over all six documents.
The extraction agent is the same one from notebook 01.

In [ ]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Extract people, places, dates, and evidence from the historical text. '
        'Be conservative: only extract what the text explicitly states. '
        'Include a short direct quote as evidence for each extracted item.'
    ),
    output_type=Extraction,
    model=DEFAULT_MODEL,
)

In [ ]:
extractions: dict[str, Extraction] = {}

for doc in DOCUMENTS:
    with trace(f"evaluation: {doc['filename']}"):
        result = await Runner.run(archive_agent, doc['text'])
    extractions[doc['filename']] = result.final_output
    print(f"done: {doc['filename']}")

print(f'\n{len(extractions)} documents extracted.')

## Baseline: exact match

Exact match after lowercasing is the strictest criterion and the right starting point.
When a score is low, the gap between the predicted set and the gold set is the exact list
of misses — nothing is hidden by a lenient match.

How to read the table:
- **High recall, low precision on a field** → the agent extracts genuine items *plus* extra noise.
- **Low recall on a field** → the agent misses items that the text contains.
- **Low scores on `letter_03`** → expected; OCR noise corrupts extracted values before they reach the matcher.

In [ ]:
all_rows = []
for filename, extraction in extractions.items():
    gold = GOLD.get(filename)
    if gold is None:
        print(f'No gold standard for {filename} — skipping.')
        continue
    df = evaluate(extraction, gold, match_fn=is_match_exact)
    df.insert(0, 'document', filename.replace('.txt', ''))
    all_rows.append(df)

results_exact = pd.concat(all_rows, ignore_index=True)
results_exact

In [ ]:
summary_exact = (
    results_exact
    .groupby('field')[['precision', 'recall', 'f1']]
    .mean()
    .round(2)
)
print('Macro-average across all documents (exact match):')
summary_exact

## Does the match criterion change our conclusions?

Before trusting the exact-match numbers, ask: are low scores genuine errors,
or a consequence of the match criterion being too strict?

The comparison below applies all three match functions to the same agent outputs.

- A **large gap between exact and substring F1** on a field means the agent is extracting
  semantically correct values that differ in surface form — a normalisation problem,
  not an extraction problem.
- A **small gap** means exact match is already fair for that field.

Look particularly at `dates`: the token-F1 strategy should close the gap for
date reformatting, but it will not fix false positives from implicit temporal references.

In [ ]:
strategies = {
    'exact':     is_match_exact,
    'substring': is_match_substring,
    'token_f1':  is_match_token_f1,
}

comparison_rows = []
for strategy_name, match_fn in strategies.items():
    for filename, extraction in extractions.items():
        gold = GOLD.get(filename)
        if gold is None:
            continue
        df = evaluate(extraction, gold, match_fn=match_fn)
        df.insert(0, 'strategy', strategy_name)
        df.insert(1, 'document', filename.replace('.txt', ''))
        comparison_rows.append(df)

comparison = pd.concat(comparison_rows, ignore_index=True)

pivot = (
    comparison
    .groupby(['strategy', 'field'])['f1']
    .mean()
    .round(2)
    .unstack('field')
    .reindex(['exact', 'substring', 'token_f1'])
)
print('Macro-average F1 by strategy and field:')
pivot

## Reading the results: systematic errors and design mismatches

The pattern that appears in most runs of this agent is **high recall, low precision on `dates`**.

The agent is instructed to extract all dates from the text, so it correctly extracts:
- The explicit letter date (`"3 September 1888"`) — gold standard expects this ✓
- Implicit temporal references (`"last autumn"`, `"winter of 1878"`, `"Tuesday"`) — gold does not ✗

**This is not a bug in the agent.** The agent follows its instructions correctly.
The low precision score reveals a *mismatch* between the agent's task definition
(extract all dates) and the gold standard's definition (extract the explicit letter date).

You have two options:

1. **Narrow the instructions** — tell the agent to extract only the explicit date written
   at the top of the letter.
2. **Expand the gold standard** — include all temporal references in the gold, reflecting
   what the agent is actually doing.

Neither is automatically correct. The right choice depends on the scholarly task.
This decision belongs in a methods section.

The cell below inspects the false-positive dates for one document to confirm the diagnosis.

In [ ]:
sample = 'letter_05_porto_1888.txt'
extraction = extractions[sample]
gold_dates = GOLD[sample]['dates']

print(f'Gold dates:      {gold_dates}')
print(f'Predicted dates: {extraction.dates}')
print()

print('True positives (TP):')
for d in extraction.dates:
    if any(is_match_exact(d, g) for g in gold_dates):
        print(f'  OK  {d!r}')

print('False positives (FP) — in the text, but not in the gold:')
for d in extraction.dates:
    if not any(is_match_exact(d, g) for g in gold_dates):
        print(f'  FP  {d!r}')

## Practical gold standard sources

A five-document gold standard is enough to test the harness, not enough to trust the results.

| Source | Notes |
|---|---|
| **Manual annotation** | Annotate a stratified sample (clean + noisy + edge cases). Use two annotators and measure inter-annotator agreement (Cohen's κ). |
| **Existing DH datasets** | CLEF HIPE provides annotated historical NER corpora across multiple languages and periods. |
| **Expert review of outputs** | Run the agent, have a domain expert correct the outputs, treat the corrected result as gold. |
| **Synthetic documents** | Generate documents with known entities for unit-testing edge cases (OCR noise, name variants, ambiguous dates). |

For a publishable DH pipeline, aim for a sample that covers the full distribution
of document types and quality levels in your collection.
Anything less is a pilot — be explicit about that in your methods section.

## Exercise

The low precision on `dates` reveals a mismatch between the agent's instructions and the
gold standard. Choose one of the following interventions and measure its effect:

**Option A — Narrow the instructions.**
Change the agent's instructions from `"Extract ... dates"` to
`"Extract only the explicit date written at the top of the letter
(the date the letter was written). Do not extract dates mentioned in the body."`.
Re-run the agent and compare precision on `dates` before and after.

**Option B — Expand the gold standard.**
Add all temporal references from `letter_05_porto_1888.txt`
(`"14th of August"`, `"winter of 1878"`, `"October"`) to the gold.
Do not re-run the agent — score the existing extractions against the new gold.
Compare recall before and after.

Either option demonstrates the same principle: **the eval score is only as meaningful
as the alignment between the task definition (instructions), the gold standard (expected outputs),
and the match criterion (grader)**. Changing any one of the three changes the score.
This is why the design choices belong in writing, not buried in code.